In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment
import logging
import os
from datetime import datetime, date as dt
from collections import defaultdict
import math
import time
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

# Configure logging
logging.basicConfig(
    filename='errors.txt',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

class ExamSeatingArrangement:
    def __init__(self, buffer=1, mode='dense', timeout=120):
        self.buffer = buffer
        self.mode = mode
        self.timeout = timeout  # Timeout in seconds for operations
        self.validate_mode()
        
        # Data storage
        self.exam_schedule = None
        self.course_roll_mapping = None
        self.roll_name_mapping = None
        self.classroom_master = None
        
        # Caches for faster lookups
        self._subject_roll_cache = {}
        self._roll_name_cache = {}
        self._date_day_cache = {}  # Cache for date to day mapping
        
        # Output containers
        self.seating_arrangement = []
        self.seats_left = []
        self.clashes = []
        
        # Progress tracking
        self.total_subjects = 0
        self.processed_subjects = 0
    
    def validate_mode(self):
        if self.mode not in ['dense', 'sparse']:
            raise ValueError("Mode must be either 'dense' or 'sparse'")
    
    def load_data(self, file_path):
        try:
            print(f"\n[1/5] Loading data from {file_path}...")
            start_time = time.time()
            
            # Use chunks for large files if needed
            sheets = pd.read_excel(file_path, sheet_name=None)
            
            # Load each sheet with progress reporting
            print("  - Loading exam schedule...")
            self.exam_schedule = sheets['in_timetable']
            
            print("  - Loading course-roll mapping...")
            self.course_roll_mapping = sheets['in_course_roll_mapping']
            
            print("  - Loading roll-name mapping...")
            self.roll_name_mapping = sheets['in_roll_name_mapping']
            
            print("  - Loading classroom data...")
            self.classroom_master = sheets['in_room_capacity']
            
            # Clean and standardize data
            print("  - Standardizing data...")
            self.standardize_data()
            
            # Pre-build caches
            print("  - Building lookup caches...")
            self._build_caches()
            
            elapsed = time.time() - start_time
            print(f"✓ Data loaded successfully in {elapsed:.2f} seconds")
            
        except Exception as e:
            logging.error(f"Error loading data from {file_path}: {str(e)}")
            raise
    
    def _build_caches(self):
        """Build caches for faster lookups."""
        # Cache for subject to rolls mapping - use sets for faster operations
        for subject, group in self.course_roll_mapping.groupby('course_code'):
            self._subject_roll_cache[subject] = group['rollno'].tolist()
        
        # Cache for roll to name mapping
        self._roll_name_cache = dict(zip(
            self.roll_name_mapping['roll'],
            self.roll_name_mapping['name']
        ))
        
        # Build date to day mapping cache
        for _, row in self.exam_schedule.iterrows():
            date = row['date']
            day = row['day']
            if date not in self._date_day_cache and pd.notna(day):
                self._date_day_cache[date] = day
                
        # Fill any missing days in the cache with weekday names
        for date in self._date_day_cache:
            if pd.isna(self._date_day_cache[date]):
                try:
                    if isinstance(date, pd.Timestamp) or isinstance(date, datetime):
                        self._date_day_cache[date] = date.strftime('%A')
                    else:
                        self._date_day_cache[date] = pd.to_datetime(date).strftime('%A')
                except:
                    self._date_day_cache[date] = 'Unknown'
    
    def standardize_data(self):
        try:
            # Standardize column names
            self.exam_schedule.columns = [col.strip().lower() for col in self.exam_schedule.columns]
            self.course_roll_mapping.columns = [col.strip().lower() for col in self.course_roll_mapping.columns]
            self.roll_name_mapping.columns = [col.strip().lower() for col in self.roll_name_mapping.columns]
            self.classroom_master.columns = [col.strip().lower() for col in self.classroom_master.columns]

            # Parse and expand the exam schedule
            self.exam_schedule = self.parse_exam_schedule(self.exam_schedule)

            # Ensure roll_no is string type and clean data
            if 'rollno' in self.course_roll_mapping.columns:
                self.course_roll_mapping['rollno'] = self.course_roll_mapping['rollno'].astype(str).str.strip()
            if 'roll' in self.roll_name_mapping.columns:
                self.roll_name_mapping['roll'] = self.roll_name_mapping['roll'].astype(str).str.strip()
            
            # Fill missing names efficiently
            if 'name' in self.roll_name_mapping.columns:
                mask = self.roll_name_mapping['name'].isna()
                self.roll_name_mapping.loc[mask, 'name'] = "Unknown Name - " + self.roll_name_mapping.loc[mask, 'roll']
            
        except Exception as e:
            logging.error(f"Error standardizing data: {str(e)}")
            raise
    
    def parse_exam_schedule(self, schedule_df):
        """Optimized exam schedule parsing."""
        parsed_rows = []
        
        for _, row in schedule_df.iterrows():
            date = row['date']
            if isinstance(date, pd.Timestamp):
                date = date.date()
            
            # Process both sessions in one loop
            for session in ['morning', 'evening']:
                if pd.notna(row[session]) and str(row[session]).strip().lower() != 'no exam':
                    subjects = [s.strip() for s in str(row[session]).split(';') if s.strip()]
                    parsed_rows.extend({
                        'date': date,
                        'day': row['day'],
                        'session': session,
                        'subject_code': subject
                    } for subject in subjects)
        
        return pd.DataFrame(parsed_rows)
    
    def select_best_room(self, available_rooms, remaining_students, current_building=None):
        """Select the most appropriate room based on mode, capacity and building proximity."""
        try:
            if available_rooms.empty:
                return None
                
            # First try to find rooms in the same building if specified
            if current_building:
                same_building_rooms = available_rooms[available_rooms['block'] == current_building]
                if not same_building_rooms.empty:
                    available_rooms = same_building_rooms
            
            if self.mode == 'dense':
                # Find smallest room that can fit all remaining students
                suitable_rooms = available_rooms[
                    available_rooms['adjusted_capacity'] >= remaining_students
                ]
                if not suitable_rooms.empty:
                    return suitable_rooms.sort_values('adjusted_capacity').iloc[0]  # Take the smallest suitable room
                else:
                    return available_rooms.sort_values('adjusted_capacity', ascending=False).iloc[0]  # Take the largest available room
            else:  # sparse mode
                return available_rooms.sort_values('adjusted_capacity', ascending=False).iloc[0]  # Take the largest available room
        except Exception as e:
            logging.error(f"Error selecting best room: {str(e)}")
            return None
    
    def detect_clashes(self):
        try:
            print("\n[2/5] Checking for exam clashes...")
            start_time = time.time()
            
            # Use groupby once and process in parallel
            grouped_exams = self.exam_schedule.groupby(['date', 'session'])
            total_groups = len(grouped_exams)
            
            # Process each group in parallel
            with ThreadPoolExecutor() as executor:
                futures = []
                for key, group in grouped_exams:
                    futures.append(executor.submit(self._process_exam_group, (key, group)))
                
                # Process results as they complete
                all_clashes = []
                for future in as_completed(futures):
                    result = future.result()
                    if result:
                        all_clashes.extend(result)
            
            self.clashes = all_clashes
            
            elapsed = time.time() - start_time
            print(f"✓ Clash detection completed in {elapsed:.2f} seconds")
            
            if self.clashes:
                print(f"   ! Found {len(self.clashes)} exam clashes!")
            else:
                print("   No exam clashes detected.")
                
            return len(self.clashes) == 0
        
        except Exception as e:
            logging.error(f"Error detecting clashes: {str(e)}")
            raise
    
    def _process_exam_group(self, group_data):
        """Process a single exam group for clash detection."""
        (date, session), exams = group_data
        subject_codes = exams['subject_code'].unique()
        clashes = []
        
        # Get all unique pairs of subjects
        for subj1, subj2 in combinations(subject_codes, 2):
            # Get roll numbers from cache
            rolls1 = set(self._subject_roll_cache.get(subj1, []))
            rolls2 = set(self._subject_roll_cache.get(subj2, []))
            
            # Find intersection
            common_rolls = rolls1.intersection(rolls2)
            
            if common_rolls:
                clashes.append({
                    'date': date,
                    'session': session,
                    'subjects': [subj1, subj2],
                    'roll_numbers': list(common_rolls)
                })
                
                # Log the clash
                for roll in common_rolls:
                    logging.error(
                        f"Roll No {roll} has multiple exams on {date}, {session} in {subj1} and {subj2}"
                    )
        
        return clashes
    
    def allocate_seats(self):
        try:
            print("\n[3/5] Allocating seats...")
            start_time = time.time()
            
            # Count total subjects to process for progress tracking
            grouped_exams = self.exam_schedule.groupby(['date', 'session'])
            self.total_subjects = 0
            for _, exams in grouped_exams:
                self.total_subjects += len(exams['subject_code'].unique())
            
            self.processed_subjects = 0
            
            # Precompute subject enrollment counts once
            subject_counts = {}
            for subject, rolls in self._subject_roll_cache.items():
                subject_counts[subject] = len(rolls)
            
            # Prepare available rooms once
            available_rooms_master = self.classroom_master.copy()
            available_rooms_master['adjusted_capacity'] = available_rooms_master['exam capacity'] - self.buffer
            available_rooms_master = available_rooms_master.sort_values(['block', 'exam capacity'], ascending=[True, False])
            
            # Process each date and session group
            for (date, session), exams in grouped_exams:
                print(f"  - Processing {date} {session}...")
                
                # Ensure the date is in our day cache
                self._ensure_date_in_cache(date)
                
                # Get subjects with student counts
                subjects_with_counts = [
                    (subj, subject_counts.get(subj, 0))
                    for subj in exams['subject_code'].unique()
                ]
                
                # Sort by student count descending
                subjects_with_counts.sort(key=lambda x: x[1], reverse=True)
                
                # Reset available rooms for this session
                available_rooms = available_rooms_master.copy()
                
                # Allocate for each subject
                for subj, student_count in subjects_with_counts:
                    sys.stdout.write(f"    - Allocating {student_count} students for {subj}...\r")
                    sys.stdout.flush()
                    
                    # Set timeout for this allocation to prevent hanging
                    allocation_start = time.time()
                    try:
                        self._allocate_subject_rooms_with_timeout(
                            subj, student_count, available_rooms.copy(), date, session
                        )
                    except Exception as e:
                        logging.error(f"Error allocating seats for {subj}: {str(e)}")
                        print(f"\n    ! Error allocating seats for {subj}: {str(e)}")
                    
                    self.processed_subjects += 1
                    sys.stdout.write(f"    - Allocated students for {subj} ({self.processed_subjects}/{self.total_subjects})\n")
            
            # Prepare seats left data
            print("  - Calculating remaining capacity...")
            self._prepare_seats_left_data()
            
            elapsed = time.time() - start_time
            print(f"✓ Seat allocation completed in {elapsed:.2f} seconds")
            
        except Exception as e:
            logging.error(f"Error allocating seats: {str(e)}")
            raise
            
    def _ensure_date_in_cache(self, date):
        """Make sure a date is in the day cache."""
        # Normalize date format to avoid mismatches
        normalized_date = None
        
        # Already in cache
        if date in self._date_day_cache:
            return
            
        # Try to standardize the date format
        try:
            if isinstance(date, str):
                normalized_date = pd.to_datetime(date).date()
            elif isinstance(date, pd.Timestamp):
                normalized_date = date.date()
            elif isinstance(date, dt):
                normalized_date = date
            else:
                normalized_date = date
                
            # Get day from exam schedule
            day_values = self.exam_schedule[self.exam_schedule['date'] == date]['day']
            if not day_values.empty:
                self._date_day_cache[date] = day_values.iloc[0]
                if normalized_date != date:
                    self._date_day_cache[normalized_date] = day_values.iloc[0]
            else:
                # Get day name from date
                weekday = pd.to_datetime(date).strftime('%A')
                self._date_day_cache[date] = weekday
                if normalized_date != date:
                    self._date_day_cache[normalized_date] = weekday
                    
        except Exception as e:
            logging.warning(f"Could not normalize date {date}: {str(e)}")
            # Store as unknown in both possible formats
            self._date_day_cache[date] = 'Unknown'
            if normalized_date is not None and normalized_date != date:
                self._date_day_cache[normalized_date] = 'Unknown'
    
    def _allocate_subject_rooms_with_timeout(self, subject_code, student_count, available_rooms, date, session):
        """Allocate rooms for a subject with timeout to prevent hanging."""
        start_time = time.time()
        
        # Create a safe copy of available rooms to modify
        available_rooms_safe = available_rooms.copy()
        
        # Get student rolls for this subject
        rolls = self._subject_roll_cache.get(subject_code, []).copy()
        if not rolls:
            logging.warning(f"No students found for {subject_code}")
            return
            
        remaining_students = len(rolls)
        current_building = None
        rooms_used = 0
        students_allocated = 0
        
        while remaining_students > 0 and not available_rooms_safe.empty:
            # Check timeout
            if time.time() - start_time > self.timeout:
                error_msg = f"Timeout reached while allocating rooms for {subject_code}"
                logging.error(error_msg)
                print(f"\n    ! {error_msg}")
                break
                
            # Select best room - use try/except to handle any issues
            try:
                room = self.select_best_room(available_rooms_safe, remaining_students, current_building)
                if room is None:
                    break
            except Exception as e:
                logging.error(f"Error selecting room for {subject_code}: {str(e)}")
                break
                    
            # Remember first building used
            if current_building is None:
                current_building = room['block']
                
            # Calculate allocation based on mode
            if self.mode == 'sparse':
                capacity = min(remaining_students, math.floor(room['adjusted_capacity'] * 0.5))
            else:
                capacity = min(remaining_students, room['adjusted_capacity'])
            
            # Make sure capacity is at least 1
            capacity = max(1, capacity)
            
            # Get actual rolls to allocate (prevent index errors)
            allocated_rolls = rolls[:min(capacity, len(rolls))]
            actual_count = len(allocated_rolls)
            
            # Get the day from our cache
            day = self._date_day_cache.get(date, 'Unknown')
            
            # Ensure we have a non-None, non-NaN value
            if day is None or pd.isna(day):
                self._ensure_date_in_cache(date)  # Try to fill cache
                day = self._date_day_cache.get(date, 'Unknown')
                
                # Final fallback
                if day is None or pd.isna(day):
                    day = 'Unknown'
            
            self.seating_arrangement.append({
                'date': date,
                'day': day,
                'session': session,
                'course_code': subject_code,
                'room': room['room no.'],
                'block': room['block'],
                'allocated_student_count': actual_count,
                'roll_list': ';'.join(allocated_rolls),
                'exam_capacity': room['exam capacity'],
                'adjusted_capacity': room['adjusted_capacity']
            })
            
            rooms_used += 1
            remaining_students -= actual_count
            students_allocated += actual_count
            rolls = rolls[actual_count:]
            
            # Update room availability - use safer approach
            try:
                # Get the index of the room
                room_idx = room.name
                
                # Update the adjusted capacity
                available_rooms_safe.at[room_idx, 'adjusted_capacity'] -= actual_count
                
                # Remove room if no more capacity
                if available_rooms_safe.at[room_idx, 'adjusted_capacity'] <= 0:
                    available_rooms_safe = available_rooms_safe.drop(room_idx)
            except Exception as e:
                logging.error(f"Error updating room capacity for {subject_code}: {str(e)}")
                # Create a new DataFrame with remaining rooms instead
                available_rooms_safe = available_rooms_safe[available_rooms_safe.index != room.name]
        
        # Log completion
        sys.stdout.write(f"      Allocated {students_allocated} students in {rooms_used} rooms\n")
        
        if remaining_students > 0:
            error_msg = f"      ! Warning: Couldn't allocate all students for {subject_code} ({remaining_students} left)"
            logging.error(error_msg)
            print(error_msg)

    def _prepare_seats_left_data(self):
        """Optimized seats left calculation."""
        try:
            if not self.seating_arrangement:
                return
                
            # Convert to DataFrame for easier processing
            seating_df = pd.DataFrame(self.seating_arrangement)
            
            # Group by room and session
            room_usage = seating_df.groupby(['room', 'block', 'exam_capacity', 'session'])['allocated_student_count'].sum().reset_index()
            
            # Calculate vacant seats
            room_usage['vacant'] = room_usage['exam_capacity'] - self.buffer - room_usage['allocated_student_count']
            room_usage['vacant'] = room_usage['vacant'].clip(lower=0)  # Ensure no negative values
            
            # Convert to output format
            self.seats_left = room_usage.rename(columns={
                'room': 'room_no',
                'allocated_student_count': 'allocated'
            }).to_dict('records')
            
        except Exception as e:
            logging.error(f"Error preparing seats left data: {str(e)}")
            raise
    
    def generate_output_files(self, output_folder='output'):
        try:
            print("\n[4/5] Generating output files...")
            start_time = time.time()
            
            # Create output directory if it doesn't exist
            os.makedirs(output_folder, exist_ok=True)
            
            # Generate overall files
            print("  - Creating overall seating arrangement file...")
            self._generate_overall_files(output_folder)
            
            # Generate per-date files
            print("  - Creating per-date files...")
            self._generate_per_date_files(output_folder)
            
            elapsed = time.time() - start_time
            print(f"✓ Output files generated in {elapsed:.2f} seconds")
            
        except Exception as e:
            logging.error(f"Error generating output files: {str(e)}")
            raise
    
    def _generate_overall_files(self, output_folder):
        """Generate overall seating arrangement files."""
        try:
            # Create overall seating arrangement dataframe
            seating_df = pd.DataFrame(self.seating_arrangement)
            if seating_df.empty:
                print("    ! No seating arrangements to output.")
                return
                
            # Select relevant columns for overall report
            overall_seating = seating_df[[
                'date', 'day', 'session', 'course_code', 'room', 'allocated_student_count', 'roll_list'
            ]]
            
            # Save to Excel
            output_path = os.path.join(output_folder, 'op_overall_seating_arrangement.xlsx')
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                overall_seating.to_excel(writer, index=False, sheet_name='Seating Plan')
                
                # Format the worksheet
                workbook = writer.book
                worksheet = writer.sheets['Seating Plan']
                worksheet.insert_rows(1)
                worksheet.cell(row=1, column=1, value="Seating Plan").font = Font(bold=True, size=14)
                worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=7)
            
            # Save seats left data
            seats_left_df = pd.DataFrame(self.seats_left)
            if not seats_left_df.empty:
                seats_left_df.to_excel(
                    os.path.join(output_folder, 'op_seats_left.xlsx'),
                    index=False
                )
            
        except Exception as e:
            logging.error(f"Error generating overall files: {str(e)}")
            raise
    
    def _generate_per_date_files(self, output_folder):
        """Generate per-date seating arrangement files."""
        try:
            if not self.seating_arrangement:
                return
                
            # Convert to DataFrame for easier processing
            seating_df = pd.DataFrame(self.seating_arrangement)
            
            # Process each date-session group
            grouped = seating_df.groupby(['date', 'session'])
            
            with ThreadPoolExecutor(max_workers=min(os.cpu_count(), 4)) as executor:
                futures = []
                
                for (date, session), group in grouped:
                    future = executor.submit(
                        self._process_date_session_group, 
                        date, session, group, output_folder
                    )
                    futures.append(future)
                
                # Monitor progress
                completed = 0
                for future in as_completed(futures):
                    try:
                        future.result()  # Get result to catch any exceptions
                        completed += 1
                        print(f"    - Completed {completed}/{len(futures)} date-session groups")
                    except Exception as e:
                        logging.error(f"Error in per-date file generation: {str(e)}")
            
        except Exception as e:
            logging.error(f"Error generating per-date files: {str(e)}")
            raise
    
    def _process_date_session_group(self, date, session, group, output_folder):
        """Process a single date-session group for file generation."""
        try:
            # Format date string
            if isinstance(date, pd.Timestamp):
                date_str = date.strftime('%d_%m_%Y')
            else:
                date_str = pd.to_datetime(date).strftime('%d_%m_%Y')
                
            session_str = session
            
            # Create folders
            session_folder = os.path.join(output_folder, date_str, session_str)
            os.makedirs(session_folder, exist_ok=True)
            
            # Process each subject-room combination
            subject_room_groups = group.groupby(['course_code', 'room'])
            
            for (subject_code, room), subject_group in subject_room_groups:
                # Generate attendance sheet
                self._generate_attendance_sheet(
                    subject_code, room, subject_group, session_folder, date, session
                )
                
                # Generate seating arrangement file
                self._generate_seating_arrangement_file(
                    subject_code, room, subject_group, session_folder
                )
                
        except Exception as e:
            logging.error(f"Error processing date-session group {date}-{session}: {str(e)}")
            raise
    
    def _generate_attendance_sheet(self, subject_code, room, subject_group, session_folder, date, session):
        """Generate attendance sheet for a specific subject and room."""
        try:
            # Get roll numbers from the first row
            first_row = subject_group.iloc[0]
            roll_list = first_row['roll_list'].split(';') if isinstance(first_row['roll_list'], str) else []
            
            # Prepare student data
            student_data = []
            for roll in roll_list:
                student_data.append({
                    'Roll No': roll,
                    'Student Name': self._roll_name_cache.get(roll, f"Unknown Name - {roll}"),
                    'Signature': ''
                })
            
            # Create Excel file
            filename = f"attendance_sheet_{subject_code}_{room}.xlsx"
            filepath = os.path.join(session_folder, filename)
            
            with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                if student_data:
                    pd.DataFrame(student_data).to_excel(writer, index=False, sheet_name='Attendance')
                else:
                    # Create empty sheet
                    pd.DataFrame(columns=['Roll No', 'Student Name', 'Signature']).to_excel(
                        writer, index=False, sheet_name='Attendance'
                    )
                
                # Format the worksheet
                workbook = writer.book
                worksheet = writer.sheets['Attendance']
                
                # Add title
                worksheet.insert_rows(1)
                title_text = f"{subject_code} - Room: {room} - Date: {date} - Session: {session}"
                cell = worksheet.cell(row=1, column=1, value=title_text)
                cell.font = Font(bold=True, size=14)
                worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=3)
                
                # Add TA and invigilator sections at the end
                start_row = len(student_data) + 4 if student_data else 4
                
                for i in range(1, 4):  # Reduced from 6 to 4 TAs for efficiency
                    worksheet.cell(row=start_row + i - 1, column=1, value=f"TA{i}:")
                
                for i in range(1, 4):  # Reduced from 6 to 4 invigilators
                    worksheet.cell(row=start_row + 3 + i, column=1, value=f"Invigilator {i}:")
            
        except Exception as e:
            logging.error(f"Error generating attendance sheet for {subject_code} in {room}: {str(e)}")
            raise
    
    def _generate_seating_arrangement_file(self, subject_code, room, subject_group, session_folder):
        """Generate seating arrangement file for a specific subject and room."""
        try:
            # Get data from the first row
            row = subject_group.iloc[0]
            
            # Filename
            filename = f"seating_arrangement_{subject_code}_{room}.xlsx"
            filepath = os.path.join(session_folder, filename)
            
            # Prepare data
            seating_data = [{
                'date': row['date'],
                'day': row['day'],
                'session': row['session'],
                'course_code': row['course_code'],
                'room': row['room'],
                'allocated_student_count': row['allocated_student_count'],
                'roll_list': row['roll_list']
            }]
            
            # Create Excel file
            with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                pd.DataFrame(seating_data).to_excel(writer, index=False, sheet_name='Seating Plan')
                
                # Format the worksheet
                workbook = writer.book
                worksheet = writer.sheets['Seating Plan']
                
                # Add title
                worksheet.insert_rows(1)
                title_cell = worksheet.cell(row=1, column=1, value="Seating Plan")
                title_cell.font = Font(bold=True, size=14)
                worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=7)
            
        except Exception as e:
            logging.error(f"Error generating seating arrangement for {subject_code} in {room}: {str(e)}")
            raise


def main():
    try:
        print("=== Exam Seating Arrangement System ===")
        
        # Get user inputs with validation and timeouts
        buffer_seats = 1
        while True:
            buffer_input = input("\nEnter number of buffer seats per classroom (default: 1): ").strip()
            if not buffer_input:
                break
            try:
                buffer_seats = int(buffer_input)
                if buffer_seats >= 0:
                    break
                print("Buffer seats must be a non-negative integer.")
            except ValueError:
                print("Please enter a valid integer for buffer seats.")
        
        mode = 'dense'
        while True:
            mode_input = input("Enter seating mode (dense/sparse, default: dense): ").strip().lower()
            if not mode_input:
                break
            if mode_input in ['dense', 'sparse']:
                mode = mode_input
                break
            print("Please enter either 'dense' or 'sparse' for seating mode.")
        
        # Get timeout value
        timeout = 60
        while True:
            timeout_input = input("Enter timeout in seconds for operations (default: 60): ").strip()
            if not timeout_input:
                break
            try:
                timeout = int(timeout_input)
                if timeout > 0:
                    break
                print("Timeout must be a positive integer.")
            except ValueError:
                print("Please enter a valid integer for timeout.")
        
        # Initialize and run system
        print("\nInitializing system...")
        seating_system = ExamSeatingArrangement(buffer=buffer_seats, mode=mode, timeout=timeout)
        
        # Get input file path
        default_path = '/home/yashbuddhadev21/Documents/IITP/IITP-Classification-Task/seating-arrangement/datasets/sorted_data.xlsx'
        input_file = input(f"Enter input file path (default: {default_path}): ").strip()
        if not input_file:
            input_file = default_path
        
        seating_system.load_data(input_file)
        
        # Check for exam clashes
        no_clashes = seating_system.detect_clashes()
        
        # Allocate seats with progress tracking
        seating_system.allocate_seats()
        
        # Generate output files
        output_folder = input("\nEnter output folder path (default: 'output'): ").strip()
        if not output_folder:
            output_folder = 'output'
            
        seating_system.generate_output_files(output_folder)
        
        print("\n[5/5] Process completed successfully!")
        print(f"Output files have been generated in the '{output_folder}' folder.")
        
    except KeyboardInterrupt:
        print("\n\nProcess interrupted. Exiting...")
    except Exception as e:
        logging.error(f"Error in main execution: {str(e)}")
        print(f"\nAn error occurred: {str(e)}. Check errors.txt for details.")


if __name__ == "__main__":
    main()

=== Exam Seating Arrangement System ===

Initializing system...

[1/5] Loading data from /home/yashbuddhadev21/Documents/IITP/IITP-Classification-Task/seating-arrangement/datasets/sorted_data.xlsx...
  - Loading exam schedule...
  - Loading course-roll mapping...
  - Loading roll-name mapping...
  - Loading classroom data...
  - Standardizing data...
  - Building lookup caches...
✓ Data loaded successfully in 0.36 seconds

[2/5] Checking for exam clashes...
✓ Clash detection completed in 0.01 seconds
   No exam clashes detected.

[3/5] Allocating seats...
  - Processing 2016-04-30 morning...
      Allocated 56 students in 2 rooms...
    - Allocated students for CB204 (1/156)
      Allocated 54 students in 2 rooms...
    - Allocated students for CB308 (2/156)
      Allocated 54 students in 2 rooms...
    - Allocated students for CE216 (3/156)
      Allocated 40 students in 1 rooms...
    - Allocated students for CS249 (4/156)
      Allocated 32 students in 1 rooms...
    - Allocated stu